# SpatialVID-HQ store preview

What one record holds, what a training window looks like, and how poses/motion attach to frames.

Set `STORE` to a store built by `datasets/prep/spatialvid_hq` (default: the local 60-clip test build). Needs the `video` dependency group (`uv sync --group video`); on macOS with Homebrew FFmpeg, start Jupyter with `DYLD_LIBRARY_PATH=/opt/homebrew/lib`.

In [ ]:
import os, json, numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from visionlab.datasets.video import VideoStore, relative_motion, camera_center, interpolate_poses

STORE = os.environ.get('SPATIALVID_STORE', '/private/tmp/claude-501/-Users-gaa019-Documents-GitHub-visionlab-datasets/65a171d3-796c-4019-adbc-e148f8f1d369/scratchpad/work/stores/spatialvid-hq-h265-640x360')
store = VideoStore(STORE)
print(len(store), 'records'); print(json.dumps(json.load(open(os.path.join(STORE, 'store_manifest.json'))), indent=1)[:600])

## Records table
`records.parquet` maps `record_idx ↔ clip_id`; splits and subsets are index sets over it.

In [ ]:
store.records.head(10)

## One record
Every field except `video` is small; arrays are ragged over the ≈5 Hz annotated frames.

In [ ]:
IDX = 0
rec = store.record(IDX)
for k, v in rec.items():
    if isinstance(v, np.ndarray): print(f'{k:16s} ndarray {v.dtype} {v.shape}')
    elif isinstance(v, dict): print(f'{k:16s} dict keys={list(v)[:6]}')
    else: print(f'{k:16s} {str(v)[:90]}')

In [ ]:
print('caption:'); print(json.dumps(rec['caption'], indent=1)[:1500])

## Frames of the clip at the annotated (pose) times

In [ ]:
fb = store.frames_at_seconds(IDX, rec['annot_frame_idx'][:12] / rec['fps'])
imgs = fb.data.permute(0, 2, 3, 1).numpy()
fig, axes = plt.subplots(2, 6, figsize=(18, 5.5))
for ax, im, fi in zip(axes.ravel(), imgs, rec['annot_frame_idx']):
    ax.imshow(im); ax.set_title(f'frame {fi}  t={fi/rec["fps"]:.2f}s', fontsize=9); ax.axis('off')
plt.suptitle(f"{rec['clip_id'][:8]} · {rec['scene_type']} · {rec['motion_tags']} · {rec['width']}x{rec['height']} @ {rec['fps']:.2f} fps, {rec['num_frames']} frames"); plt.tight_layout()

## Camera trajectory
Stored poses are world→camera; camera centers are `-Rᵀt`. Scale is not metric.

In [ ]:
C = np.stack([camera_center(p) for p in rec['poses']])
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(C[:, 0], C[:, 2], '-o', ms=3); ax[0].set_xlabel('x'); ax[0].set_ylabel('z (forward)'); ax[0].set_title('top view of camera centers'); ax[0].axis('equal')
rel = np.stack([relative_motion(rec['poses'][i], rec['poses'][i+1]) for i in range(len(rec['poses'])-1)])
ax[1].plot(rel[:, :3]); ax[1].legend(['dx right', 'dy down', 'dz forward']); ax[1].set_title('relative translation per annotated step (camera-t axes)'); ax[1].set_xlabel('annotation step')
plt.tight_layout()
print('mean |rot| deg per step:', np.degrees(np.abs(rel[:, 3:]).mean(0)).round(2))

## A training sample: window spec → tensors
A sample is `(record_idx, start_frame, T, stride)`. Frames come from one decode of ≤1 s of video (keyframe every second); poses are interpolated to the exact frames (linear t, slerp q); relative motion is what the motion-conditioned model receives.

In [ ]:
def sample(store, idx, start, T=8, stride=3):
    rec = store.record(idx)
    frames = store.frames(idx, start=start, count=T, stride=stride)          # FrameBatch: uint8 [T,3,H,W] + pts
    fidx = np.arange(start, start + T*stride, stride)
    poses = store.poses_at(idx, fidx, rec=rec)                                # [T,7]
    motion = np.stack([relative_motion(poses[i], poses[i+1]) for i in range(T-1)])   # [T-1,6]
    return dict(video=frames.data, t_sec=frames.pts_seconds, frame_idx=fidx, poses=poses, motion=motion,
                clip_id=rec['clip_id'], fps=rec['fps'])

s = sample(store, IDX, start=int(rec['fps']*1.0), T=8, stride=3)
print({k: (tuple(v.shape) if hasattr(v, 'shape') else v) for k, v in s.items()})
fig, axes = plt.subplots(1, 8, figsize=(20, 2.4))
for ax, im, fi, m in zip(axes, s['video'].permute(0,2,3,1).numpy(), s['frame_idx'], np.vstack([np.zeros(6), s['motion']])):
    ax.imshow(im); ax.axis('off'); ax.set_title(f'f{fi}\ndz={m[2]:+.3f} ry={np.degrees(m[4]):+.1f}°', fontsize=8)
plt.tight_layout()

## Batching
Fixed T makes batches plain stacks: `[B,T,3,H,W]`, `[B,T,7]`, `[B,T-1,6]`. Anchors are valid when `start + (T-1)·stride < num_frames`. The 'without motion' control is the identical batch with `motion` zeroed.

In [ ]:
import time
rng = np.random.default_rng(0)
T, stride, B = 8, 3, 4
t0 = time.perf_counter(); batch = []
for i in rng.choice(len(store), B, replace=False):
    n = int(store.raw(int(i), 'num_frames')); start = int(rng.integers(0, max(1, n - (T-1)*stride)))
    batch.append(sample(store, int(i), start, T, stride))
video = torch.stack([b['video'] for b in batch]); poses = np.stack([b['poses'] for b in batch]); motion = np.stack([b['motion'] for b in batch])
print('video', tuple(video.shape), video.dtype, '| poses', poses.shape, '| motion', motion.shape, f'| {(time.perf_counter()-t0)*1000:.0f} ms for B={B} on CPU (first-touch decoders)')
fig, axes = plt.subplots(B, T, figsize=(2.2*T, 1.6*B))
for r, b in enumerate(batch):
    for c in range(T):
        axes[r, c].imshow(b['video'][c].permute(1,2,0).numpy()); axes[r, c].axis('off')
    axes[r, 0].set_title(b['clip_id'][:8], fontsize=8, loc='left')
plt.tight_layout()

## Through SlipstreamLoader (slipstream ≥ 0.6.0)
The production path: the `video` bytes field is the primary (zero-copy prefetch banks), secondary bytes fields (`poses`, `intrinsics`, `annot_frame_idx`) arrive as owned `{data, sizes}` copies, scalars/strings as tensors/lists. `indices=` selects a split or subset without touching the store; `warmup_cache()` then reads only those records.

In [ ]:
import io, pathlib
from slipstream import SlipstreamLoader

class StoreDataset:            # minimal handle slipstream needs to open an existing cache
    cache_path = pathlib.Path(STORE); dataset_hash = 'spatialvid-hq'

idx = store.records.record_idx.to_numpy()[::2]           # e.g. a split's record indices
loader = SlipstreamLoader(StoreDataset(), batch_size=8, shuffle=True, seed=0, indices=idx, verbose=False, drop_last=False)
print('primary field:', loader.image_field, '| warmup:', {k: v for k, v in loader.warmup_cache(verbose=False).items() if k in ('subset', 'num_records', 'num_ranges', 'total_bytes')})
batch = next(iter(loader)); loader.shutdown()
v, p = batch['video'], batch['poses']
print('video bank view', v['data'].shape, 'sizes', v['sizes'][:4].tolist())
print('poses payload  ', p['data'].shape, 'sizes', p['sizes'][:4].tolist())
poses0 = np.load(io.BytesIO(bytes(p['data'][0, :int(p['sizes'][0])])))
print('clip', batch['clip_id'][0][:8], 'fps', float(batch['fps'][0]), 'num_frames', int(batch['num_frames'][0]), '| poses', poses0.shape, poses0.dtype)
# decode a window straight from the bank view (torchcodec accepts bytes)
from visionlab.datasets.video import import_torchcodec
VideoDecoder = import_torchcodec()
fb = VideoDecoder(bytes(v['data'][0, :int(v['sizes'][0])])).get_frames_in_range(0, 8)
print('decoded from batch bytes:', tuple(fb.data.shape))

## Dataset-level metadata
Join `records.parquet` with `index/clips.parquet` (and a `splits/<version>.parquet`) for filtering; the store never changes when a split does.

In [ ]:
import pathlib
work = pathlib.Path(STORE).parents[1]
clips = pd.read_parquet(work / 'index' / 'clips.parquet')
df = store.records.merge(clips, on=['clip_id', 'group_id'], how='left')
splits = sorted((work / 'splits').glob('*.parquet'))
if splits:
    sp = pd.read_parquet(splits[-1], columns=['clip_id', 'split']); df = df.merge(sp, on='clip_id', how='left'); print('split file:', splits[-1].name)
df[['record_idx', 'clip_id', 'source_id', 'scene_type', 'motion_tags', 'time_of_day', 'weather', 'crowd_density', 'duration_s', 'n_annot'] + (['split'] if splits else [])].head(12)